# Lab 2: Multi-Input Model

## Topic 3: Flexible ML Architectures with Vibe Coding

In this lab, we build a **multi-input model** that combines image data with synthetic metadata features. This demonstrates a powerful pattern: fusing heterogeneous data sources to improve predictions.

### What You Will Learn
- Designing multi-input architectures with separate processing branches
- Generating synthetic metadata features from images
- Training with dictionary-style inputs
- Comparing image-only vs image+metadata performance

In [ ]:
# Run this cell in Google Colab to install dependencies
# Skip if running locally with uv
import sys
if 'google.colab' in sys.modules:
    !pip install -q keras torch torchvision gradio python-dotenv datasets transformers huggingface_hub
    print('Dependencies installed!')

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import keras
import numpy as np
import matplotlib.pyplot as plt

from keras import layers, Model, Input
from keras.utils import plot_model

print(f"Keras version: {keras.__version__}")
print(f"Keras backend: {keras.backend.backend()}")

## 1. Load CIFAR-10 and Generate Synthetic Metadata

In [ ]:
# Load CIFAR-10
(X_train_img, y_train), (X_test_img, y_test) = keras.datasets.cifar10.load_data()

# Normalize pixel values
X_train_img = X_train_img.astype("float32") / 255.0
X_test_img = X_test_img.astype("float32") / 255.0

CLASS_NAMES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print(f"Training images: {X_train_img.shape}")
print(f"Test images: {X_test_img.shape}")

In [ ]:
def extract_metadata(images):
    """Extract synthetic metadata features from images.
    
    Features:
    - Mean brightness (average pixel value across all channels)
    - Standard deviation of pixel values
    - Edge count (sum of gradient magnitudes using Sobel-like approximation)
    """
    metadata = []
    
    for img in images:
        # Convert to grayscale for some features
        gray = np.mean(img, axis=-1)
        
        # Feature 1: Mean brightness
        mean_brightness = np.mean(img)
        
        # Feature 2: Standard deviation of pixel values
        std_dev = np.std(img)
        
        # Feature 3: Edge count (simple gradient magnitude)
        # Compute horizontal and vertical gradients
        grad_x = np.abs(np.diff(gray, axis=1))  # Horizontal edges
        grad_y = np.abs(np.diff(gray, axis=0))  # Vertical edges
        edge_count = np.mean(grad_x) + np.mean(grad_y)
        
        metadata.append([mean_brightness, std_dev, edge_count])
    
    return np.array(metadata, dtype="float32")

# Generate metadata for train and test sets
print("Extracting metadata from training images...")
X_train_meta = extract_metadata(X_train_img)
print("Extracting metadata from test images...")
X_test_meta = extract_metadata(X_test_img)

print(f"\nTraining metadata shape: {X_train_meta.shape}")
print(f"Test metadata shape: {X_test_meta.shape}")
print(f"\nSample metadata (first 5):")
print(f"  Mean brightness: {X_train_meta[:5, 0]}")
print(f"  Std deviation:   {X_train_meta[:5, 1]}")
print(f"  Edge count:      {X_train_meta[:5, 2]}")

In [ ]:
# Visualize metadata distribution by class
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
feature_names = ["Mean Brightness", "Std Deviation", "Edge Count"]

for i, (ax, fname) in enumerate(zip(axes, feature_names)):
    for cls_idx in range(10):
        mask = y_train.flatten() == cls_idx
        ax.hist(X_train_meta[mask, i], bins=30, alpha=0.3, label=CLASS_NAMES[cls_idx])
    ax.set_title(fname)
    ax.set_xlabel("Value")
    ax.set_ylabel("Frequency")

axes[2].legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
plt.suptitle("Metadata Feature Distributions by Class", fontsize=14)
plt.tight_layout()
plt.show()

## 2. Image-Only Model (Baseline)

First, let's build and train a model that only uses image data as our baseline.

In [ ]:
# Image-only baseline model
img_input = Input(shape=(32, 32, 3), name="image")

x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(img_input)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(64, activation="relu")(x)
outputs = layers.Dense(10, activation="softmax", name="classification")(x)

image_only_model = Model(inputs=img_input, outputs=outputs, name="image_only_model")
image_only_model.summary()

In [ ]:
# Train image-only model
image_only_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

image_only_history = image_only_model.fit(
    {"image": X_train_img}, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

img_loss, img_acc = image_only_model.evaluate(
    {"image": X_test_img}, y_test, verbose=0
)
print(f"\nImage-Only Model - Test Loss: {img_loss:.4f}, Test Accuracy: {img_acc:.4f}")

## 3. Multi-Input Model (Image + Metadata)

Now we build the multi-input model with two branches:
- **Image branch**: Conv2D pipeline ending with GlobalAveragePooling2D
- **Metadata branch**: Dense network processing the 3 metadata features

Both branches are merged via `Concatenate` and fed into a final Dense classification head.

In [ ]:
# === Image Branch ===
image_input = Input(shape=(32, 32, 3), name="image")

img_x = layers.Conv2D(32, (3, 3), activation="relu", padding="same", name="img_conv1")(image_input)
img_x = layers.MaxPooling2D((2, 2), name="img_pool1")(img_x)
img_x = layers.Conv2D(64, (3, 3), activation="relu", padding="same", name="img_conv2")(img_x)
img_x = layers.MaxPooling2D((2, 2), name="img_pool2")(img_x)
img_x = layers.Conv2D(64, (3, 3), activation="relu", padding="same", name="img_conv3")(img_x)
img_x = layers.GlobalAveragePooling2D(name="img_global_pool")(img_x)

# === Metadata Branch ===
meta_input = Input(shape=(3,), name="meta")

meta_x = layers.Dense(32, activation="relu", name="meta_dense1")(meta_input)
meta_x = layers.Dense(16, activation="relu", name="meta_dense2")(meta_x)

# === Merge Branches ===
merged = layers.Concatenate(name="merge")([img_x, meta_x])

# === Classification Head ===
x = layers.Dense(64, activation="relu", name="head_dense1")(merged)
x = layers.Dropout(0.3, name="head_dropout")(x)
outputs = layers.Dense(10, activation="softmax", name="classification")(x)

multi_input_model = Model(
    inputs={"image": image_input, "meta": meta_input},
    outputs=outputs,
    name="multi_input_model"
)

multi_input_model.summary()

In [ ]:
# Visualize the multi-input architecture
plot_model(
    multi_input_model,
    show_shapes=True,
    show_layer_names=True,
    to_file="multi_input_model.png",
    dpi=100
)

In [ ]:
# Train the multi-input model with dict-style input
multi_input_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

multi_history = multi_input_model.fit(
    {"image": X_train_img, "meta": X_train_meta},
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

In [ ]:
# Evaluate the multi-input model
multi_loss, multi_acc = multi_input_model.evaluate(
    {"image": X_test_img, "meta": X_test_meta},
    y_test, verbose=0
)
print(f"Multi-Input Model - Test Loss: {multi_loss:.4f}, Test Accuracy: {multi_acc:.4f}")

## 4. Compare Image-Only vs Multi-Input

In [ ]:
# Compare training histories
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(image_only_history.history["val_accuracy"], label="Image Only", linestyle="--")
axes[0].plot(multi_history.history["val_accuracy"], label="Image + Metadata")
axes[0].set_title("Validation Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(image_only_history.history["val_loss"], label="Image Only", linestyle="--")
axes[1].plot(multi_history.history["val_loss"], label="Image + Metadata")
axes[1].set_title("Validation Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("Image-Only vs Multi-Input Model", fontsize=14)
plt.tight_layout()
plt.show()

# Summary
print("\n" + "="*55)
print(f"{'Model':<25} {'Test Loss':<15} {'Test Accuracy':<15}")
print("="*55)
print(f"{'Image Only':<25} {img_loss:<15.4f} {img_acc:<15.4f}")
print(f"{'Image + Metadata':<25} {multi_loss:<15.4f} {multi_acc:<15.4f}")
print("="*55)
print(f"\nAccuracy difference: {(multi_acc - img_acc)*100:+.2f}%")

## 5. Gradio Interface

Upload an image and enter metadata values to get predictions from the multi-input model.

In [ ]:
import gradio as gr

def extract_single_metadata(img_array):
    """Extract metadata from a single preprocessed image."""
    gray = np.mean(img_array, axis=-1)
    mean_brightness = float(np.mean(img_array))
    std_dev = float(np.std(img_array))
    grad_x = np.abs(np.diff(gray, axis=1))
    grad_y = np.abs(np.diff(gray, axis=0))
    edge_count = float(np.mean(grad_x) + np.mean(grad_y))
    return mean_brightness, std_dev, edge_count

def classify_multi_input(image, mean_brightness, std_dev, edge_count, auto_extract):
    """Classify an image using the multi-input model."""
    if image is None:
        return {name: 0.0 for name in CLASS_NAMES}
    
    import PIL.Image
    img = PIL.Image.fromarray(image).resize((32, 32))
    img_array = np.array(img).astype("float32") / 255.0
    
    # Auto-extract metadata from the image if checkbox is checked
    if auto_extract:
        mean_brightness, std_dev, edge_count = extract_single_metadata(img_array)
    
    # Prepare inputs
    img_batch = np.expand_dims(img_array, axis=0)
    meta_batch = np.array([[mean_brightness, std_dev, edge_count]], dtype="float32")
    
    # Predict
    predictions = multi_input_model.predict(
        {"image": img_batch, "meta": meta_batch}, verbose=0
    )[0]
    
    return {CLASS_NAMES[i]: float(predictions[i]) for i in range(10)}

# Create Gradio interface
demo = gr.Interface(
    fn=classify_multi_input,
    inputs=[
        gr.Image(label="Upload an Image"),
        gr.Number(label="Mean Brightness", value=0.5, minimum=0.0, maximum=1.0),
        gr.Number(label="Std Deviation", value=0.2, minimum=0.0, maximum=1.0),
        gr.Number(label="Edge Count", value=0.1, minimum=0.0, maximum=1.0),
        gr.Checkbox(label="Auto-extract metadata from image", value=True)
    ],
    outputs=gr.Label(num_top_classes=5, label="Predictions"),
    title="CIFAR-10 Multi-Input Classifier",
    description=(
        "Upload an image and optionally provide metadata values (mean brightness, "
        "std deviation, edge count). Check 'Auto-extract' to compute metadata automatically "
        "from the uploaded image."
    ),
    flagging_mode="never"
)

demo.launch()